#Coding Exercise: Autoencoders, VAEs, and GANs


**Topics:** Autoencoder, Variational Autoencoder (VAE), Generative Adversarial Network (GAN)

---

## Overview

In this exercise, you will implement three fundamental generative/representation learning models:

1. **Part 1: Autoencoder** - Learn compressed representations of data
2. **Part 2: Variational Autoencoder (VAE)** - Learn probabilistic latent spaces with generation capability
3. **Part 3: Generative Adversarial Network (GAN)** - Generate new samples through adversarial training

We will use the **MNIST dataset** (handwritten digits) for all exercises.

---

## Learning Objectives

After completing this exercise, you will be able to:
- Build encoder-decoder architectures in PyTorch
- Understand and implement the reparameterization trick for VAEs
- Implement the VAE loss function (reconstruction + KL divergence)
- Build and train GANs with generator and discriminator networks
- Visualize latent spaces and generated samples

---

## Instructions

- Look for **`# TODO:`** comments - these indicate where you need to write code
- Each TODO section has hints to guide you
- Run the provided test cells to verify your implementation
- Feel free to ask questions if you get stuck!

---

## 🔧 Setup and Imports

Run this cell to install and import all necessary libraries.

In [ ]:
# Install dependencies
!pip install torch torchvision matplotlib numpy -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 📊 Load MNIST Dataset

We'll use the MNIST dataset which contains 28x28 grayscale images of handwritten digits (0-9).

In [ ]:
# Data loading
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Download and load training data
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.suptitle('Sample MNIST Images', fontsize=14)
plt.tight_layout()
plt.show()

---

# Part 1: Autoencoder 🔄

## 1.1 Background

An **Autoencoder** consists of two parts:
- **Encoder**: Compresses the input into a lower-dimensional latent representation
- **Decoder**: Reconstructs the input from the latent representation

```
Input (784) → Encoder → Latent Space (e.g., 32) → Decoder → Output (784)
```

The model is trained to minimize the **reconstruction loss** between input and output.

## 1.2 Architecture

We'll build a simple fully-connected autoencoder:
- **Encoder**: 784 → 256 → 128 → 32 (latent dimension)
- **Decoder**: 32 → 128 → 256 → 784

---

## Exercise 1.1: Implement the Encoder

Complete the `Encoder` class below. The encoder should:
1. Take a flattened image (784 dimensions) as input
2. Pass through fully connected layers: 784 → 256 → 128 → latent_dim
3. Use ReLU activation after each hidden layer (not after the final layer)

PS: You can also define your own architecture with different number of neurons.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim1=256, hidden_dim2=128, latent_dim=32):
        super(Encoder, self).__init__()

        # TODO: Define the encoder layers
        # Hint: Use nn.Linear for fully connected layers
        # Architecture: input_dim → hidden_dim1 → hidden_dim2 → latent_dim

        self.fc1 = None  # TODO: First layer (input_dim → hidden_dim1)
        self.fc2 = None  # TODO: Second layer (hidden_dim1 → hidden_dim2)
        self.fc3 = None  # TODO: Third layer (hidden_dim2 → latent_dim)

    def forward(self, x):
        # TODO: Implement the forward pass
        # Hint: Apply ReLU activation after fc1 and fc2, but NOT after fc3
        # Use F.relu() or torch.relu()

        pass  # TODO: Replace with your implementation


# Test your encoder, you can change the latent_dim size, and check with test_output.shape == (1, your number)
test_encoder = Encoder()
test_input = torch.randn(1, 784)
try:
    test_output = test_encoder(test_input)
    assert test_output.shape == (1, 32), f"Expected shape (1, 32), got {test_output.shape}"
    print("✅ Encoder test passed! Output shape:", test_output.shape)
except Exception as e:
    print(f"❌ Encoder test failed: {e}")

## Exercise 1.2: Implement the Decoder

Complete the `Decoder` class below. The decoder should:
1. Take the latent representation (32 dimensions) as input
2. Pass through fully connected layers: latent_dim → 128 → 256 → 784
3. Use ReLU activation after hidden layers
4. Use **Sigmoid** activation at the output (to get values between 0 and 1)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=32, hidden_dim1=128, hidden_dim2=256, output_dim=784):
        super(Decoder, self).__init__()

        # TODO: Define the decoder layers
        # Architecture: latent_dim → hidden_dim1 → hidden_dim2 → output_dim

        self.fc1 = None  # TODO: First layer (latent_dim → hidden_dim1)
        self.fc2 = None  # TODO: Second layer (hidden_dim1 → hidden_dim2)
        self.fc3 = None  # TODO: Third layer (hidden_dim2 → output_dim)

    def forward(self, z):
        # TODO: Implement the forward pass
        # Hint: Apply ReLU after fc1 and fc2
        # Apply Sigmoid after fc3 (use torch.sigmoid())

        pass  # TODO: Replace with your implementation


# Test your decoder
test_decoder = Decoder()
test_latent = torch.randn(1, 32)
try:
    test_output = test_decoder(test_latent)
    assert test_output.shape == (1, 784), f"Expected shape (1, 784), got {test_output.shape}"
    assert test_output.min() >= 0 and test_output.max() <= 1, "Output should be between 0 and 1 (Sigmoid)"
    print("✅ Decoder test passed! Output shape:", test_output.shape)
except Exception as e:
    print(f"❌ Decoder test failed: {e}")

## Exercise 1.3: Combine into Autoencoder

Now combine the encoder and decoder into a complete autoencoder.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super(Autoencoder, self).__init__()

        # TODO: Create encoder and decoder instances
        self.encoder = None  # TODO: Create Encoder with latent_dim
        self.decoder = None  # TODO: Create Decoder with latent_dim

    def forward(self, x):
        # TODO: Implement forward pass
        # Step 1: Flatten the input image (batch_size, 1, 28, 28) → (batch_size, 784)
        # Step 2: Pass through encoder to get latent representation
        # Step 3: Pass latent through decoder to get reconstruction
        # Step 4: Reshape back to image format (batch_size, 1, 28, 28)

        pass  # TODO: Replace with your implementation

    def encode(self, x):
        """Get latent representation"""
        x = x.view(x.size(0), -1)
        return self.encoder(x)


# Test autoencoder
test_ae = Autoencoder()
test_input = torch.randn(4, 1, 28, 28)
try:
    test_output = test_ae(test_input)
    assert test_output.shape == (4, 1, 28, 28), f"Expected shape (4, 1, 28, 28), got {test_output.shape}"
    print("✅ Autoencoder test passed! Output shape:", test_output.shape)
except Exception as e:
    print(f"❌ Autoencoder test failed: {e}")

## Exercise 1.4: Training Loop

Complete the training function. We use **Mean Squared Error (MSE)** as the reconstruction loss.

In [ ]:
def train_autoencoder(model, train_loader, epochs=10, lr=1e-3):
    """
    Train the autoencoder model.

    Args:
        model: Autoencoder model
        train_loader: DataLoader for training data
        epochs: Number of training epochs
        lr: Learning rate
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # TODO: Define the loss function
    # Hint: Use nn.MSELoss() for reconstruction loss
    criterion = None  # TODO: Define loss function

    train_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0

        for batch_idx, (data, _) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')):
            data = data.to(device)

            # TODO: Complete the training step
            # Step 1: Zero the gradients
            # Step 2: Forward pass (get reconstruction)
            # Step 3: Compute loss between reconstruction and original
            # Step 4: Backward pass
            # Step 5: Update weights

            # TODO: Your code here

            pass  # TODO: Remove this and implement the training step

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}')

    return train_losses

In [ ]:
# Create and train the autoencoder
autoencoder = Autoencoder(latent_dim=32)
ae_losses = train_autoencoder(autoencoder, train_loader, epochs=50, lr=1e-3) # one can increase the epochs and add early stop condition

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(ae_losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Autoencoder Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## Exercise 1.5: Visualize Reconstructions

Let's see how well the autoencoder reconstructs the input images.

In [ ]:
def visualize_reconstructions(model, test_loader, n_samples=8):
    """Visualize original vs reconstructed images"""
    model.eval()

    # Get a batch of test data
    data, labels = next(iter(test_loader))
    data = data[:n_samples].to(device)

    with torch.no_grad():
        reconstructions = model(data)

    # Plot
    fig, axes = plt.subplots(2, n_samples, figsize=(16, 4))

    for i in range(n_samples):
        # Original
        axes[0, i].imshow(data[i].cpu().squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Original', fontsize=12)

        # Reconstruction
        axes[1, i].imshow(reconstructions[i].cpu().squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Reconstruction', fontsize=12)

    plt.suptitle('Autoencoder: Original vs Reconstruction', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_reconstructions(autoencoder, test_loader)

What happens if you reduce the latent dimension from 32 to 2? Try it and observe the reconstruction quality.

---

# Part 2: Variational Autoencoder (VAE)

## 2.1 Background

A **Variational Autoencoder** differs from a standard autoencoder:

- Instead of encoding to a fixed latent vector, the encoder outputs **mean (μ)** and **log-variance (log σ²)** of a Gaussian distribution
- We sample from this distribution using the **reparameterization trick**
- The loss has two components:
  1. **Reconstruction Loss**: How well we reconstruct the input
  2. **KL Divergence**: Regularizes the latent space to be close to N(0, 1)

## 2.2 The Reparameterization Trick

To backpropagate through sampling, we use:

```
z = μ + σ * ε,  where ε ~ N(0, 1)
```

This moves the randomness to ε, allowing gradients to flow through μ and σ.

---

## Exercise 2.1: Implement VAE Encoder

The VAE encoder outputs **two vectors**: mean (μ) and log-variance (log σ²).

In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=32):
        super(VAEEncoder, self).__init__()

        # Shared layers # you can have more linear layers
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 128)

        # TODO: Define two separate output layers
        # One for mean (mu) and one for log-variance (logvar)
        # Both should map from 128 → latent_dim

        self.fc_mu = None      # TODO: Layer for mean
        self.fc_logvar = None  # TODO: Layer for log-variance

    def forward(self, x):
        # Shared encoding path
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))

        # TODO: Compute mu and logvar
        # Hint: Pass h through the respective layers (no activation needed)

        mu = None      # TODO: Compute mean
        logvar = None  # TODO: Compute log-variance

        return mu, logvar


# Test VAE encoder
test_vae_enc = VAEEncoder()
test_input = torch.randn(4, 784)
try:
    mu, logvar = test_vae_enc(test_input)
    assert mu.shape == (4, 32), f"Expected mu shape (4, 32), got {mu.shape}"
    assert logvar.shape == (4, 32), f"Expected logvar shape (4, 32), got {logvar.shape}"
    print("✅ VAE Encoder test passed!")
    print(f"   mu shape: {mu.shape}, logvar shape: {logvar.shape}")
except Exception as e:
    print(f"❌ VAE Encoder test failed: {e}")

## Exercise 2.2: Implement the Reparameterization Trick

This is the key innovation of VAEs! Implement the function to sample from the latent distribution.

In [ ]:
def reparameterize(mu, logvar):
    """
    Reparameterization trick: sample z from N(mu, sigma^2)

    The trick: z = mu + sigma * epsilon, where epsilon ~ N(0, 1)

    Args:
        mu: Mean of the latent distribution (batch_size, latent_dim)
        logvar: Log variance of the latent distribution (batch_size, latent_dim)

    Returns:
        z: Sampled latent vector (batch_size, latent_dim)
    """
    # TODO: Implement the reparameterization trick
    # Step 1: Compute standard deviation from log-variance
    #         std = exp(0.5 * logvar)  [because logvar = log(sigma^2) = 2*log(sigma)]
    # Step 2: Sample epsilon from standard normal N(0, 1)
    #         Use torch.randn_like(std)
    # Step 3: Compute z = mu + std * epsilon

    std = None  # TODO: Compute standard deviation
    eps = None  # TODO: Sample epsilon
    z = None    # TODO: Compute z

    return z


# Test reparameterization
test_mu = torch.zeros(4, 32)
test_logvar = torch.zeros(4, 32)  # std = 1
try:
    test_z = reparameterize(test_mu, test_logvar)
    assert test_z.shape == (4, 32), f"Expected shape (4, 32), got {test_z.shape}"
    # With mu=0, logvar=0 (std=1), z should be roughly N(0,1)
    print("✅ Reparameterization test passed!")
    print(f"   Sample mean: {test_z.mean():.3f} (should be ~0)")
    print(f"   Sample std: {test_z.std():.3f} (should be ~1)")
except Exception as e:
    print(f"❌ Reparameterization test failed: {e}")

## Exercise 2.3: Complete VAE Model

Now combine everything into a complete VAE model.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=32):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder = VAEEncoder(latent_dim=latent_dim)

        # Decoder (reuse from Part 1)
        self.decoder = Decoder(latent_dim=latent_dim)

    def encode(self, x):
        """Encode input to latent distribution parameters"""
        x = x.view(x.size(0), -1)
        return self.encoder(x)

    def decode(self, z):
        """Decode latent vector to reconstruction"""
        return self.decoder(z)

    def forward(self, x):
        # TODO: Implement the forward pass
        # Step 1: Encode to get mu and logvar
        # Step 2: Sample z using reparameterization trick
        # Step 3: Decode z to get reconstruction
        # Step 4: Reshape to image format
        # Return: reconstruction, mu, logvar (all three are needed for the loss)

        mu, logvar = None, None  # TODO: Encode
        z = None                  # TODO: Reparameterize
        reconstruction = None     # TODO: Decode

        return reconstruction.view(-1, 1, 28, 28), mu, logvar

    def sample(self, n_samples):
        """Generate new samples by sampling from the prior N(0, 1)"""
        z = torch.randn(n_samples, self.latent_dim).to(next(self.parameters()).device)
        samples = self.decode(z)
        return samples.view(-1, 1, 28, 28)


# Test VAE
test_vae = VAE()
test_input = torch.randn(4, 1, 28, 28)
try:
    recon, mu, logvar = test_vae(test_input)
    assert recon.shape == (4, 1, 28, 28), f"Expected recon shape (4, 1, 28, 28), got {recon.shape}"
    assert mu.shape == (4, 32), f"Expected mu shape (4, 32), got {mu.shape}"
    print("✅ VAE test passed!")
except Exception as e:
    print(f"❌ VAE test failed: {e}")

## Exercise 2.4: Implement VAE Loss Function

The VAE loss consists of two terms:

1. **Reconstruction Loss**: Binary Cross-Entropy between input and reconstruction
2. **KL Divergence**: Measures how far the learned distribution is from N(0, 1)

$$\mathcal{L} = \mathbb{E}[\log p(x|z)] - D_{KL}(q(z|x) || p(z))$$

The KL divergence for Gaussians has a closed form:

$$D_{KL} = -\frac{1}{2} \sum (1 + \log(\sigma^2) - \mu^2 - \sigma^2)$$

In [ ]:
def vae_loss(reconstruction, x, mu, logvar):
    """
    Compute VAE loss = Reconstruction Loss + KL Divergence

    Args:
        reconstruction: Reconstructed images (batch_size, 1, 28, 28)
        x: Original images (batch_size, 1, 28, 28)
        mu: Mean of latent distribution (batch_size, latent_dim)
        logvar: Log variance of latent distribution (batch_size, latent_dim)

    Returns:
        total_loss: Combined loss
        recon_loss: Reconstruction loss only (for logging)
        kl_loss: KL divergence only (for logging)
    """
    batch_size = x.size(0)

    # TODO: Compute Reconstruction Loss
    # Use Binary Cross Entropy: F.binary_cross_entropy(reconstruction, x, reduction='sum') or recudtion="mean"
    # Divide by batch_size to get average loss per sample

    recon_loss = None  # TODO: Implement reconstruction loss

    # TODO: Compute KL Divergence
    # Formula: -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    # Sum over latent dimensions, then average over batch
    # Hint: Use torch.sum() and torch.exp()

    kl_loss = None  # TODO: Implement KL divergence

    total_loss = recon_loss + kl_loss

    return total_loss, recon_loss, kl_loss


# Test VAE loss
test_x = torch.rand(4, 1, 28, 28)  # Random images in [0, 1]
test_recon = torch.rand(4, 1, 28, 28)
test_mu = torch.randn(4, 32)
test_logvar = torch.randn(4, 32)

try:
    total, recon, kl = vae_loss(test_recon, test_x, test_mu, test_logvar)
    assert total is not None and recon is not None and kl is not None
    print("✅ VAE loss test passed!")
    print(f"   Total: {total:.2f}, Recon: {recon:.2f}, KL: {kl:.2f}")
except Exception as e:
    print(f"❌ VAE loss test failed: {e}")

## Exercise 2.5: Train the VAE

In [ ]:
def train_vae(model, train_loader, epochs=10, lr=1e-3):
    """Train the VAE model"""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        epoch_recon = 0
        epoch_kl = 0

        for batch_idx, (data, _) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')):
            data = data.to(device)

            optimizer.zero_grad()

            # Forward pass
            reconstruction, mu, logvar = model(data)

            # Compute loss
            loss, recon_loss, kl_loss = vae_loss(reconstruction, data, mu, logvar)

            # Backward pass
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()

        n_batches = len(train_loader)
        train_losses.append(epoch_loss / n_batches)
        print(f'Epoch {epoch+1}: Loss={epoch_loss/n_batches:.2f}, '
              f'Recon={epoch_recon/n_batches:.2f}, KL={epoch_kl/n_batches:.2f}')

    return train_losses

In [ ]:
# Train the VAE
vae = VAE(latent_dim=32)
vae_losses = train_vae(vae, train_loader, epochs=50, lr=1e-3)

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(vae_losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## Exercise 2.6: Generate New Samples

One advantage of VAEs over regular autoencoders is that we can generate new samples!

In [ ]:
def visualize_vae_samples(model, n_samples=16):
    """Generate and visualize new samples from the VAE"""
    model.eval()

    with torch.no_grad():
        samples = model.sample(n_samples)

    # Plot
    n_rows = int(np.sqrt(n_samples))
    n_cols = n_samples // n_rows

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 12))

    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i].cpu().squeeze(), cmap='gray')
        ax.axis('off')

    plt.suptitle('VAE Generated Samples', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_vae_samples(vae, n_samples=16)

##Visualize Latent Space (2D)


For high-dimensional latent spaces (like 32D), we use **dimensionality reduction** to visualize.

Visualize 32D latent space using PCA or t-SNE

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

def plot_latent_space_highdim(model, test_loader, method='tsne'):
    """Visualize high-dimensional latent space using PCA or t-SNE"""
    model.eval()

    latents = []
    labels_list = []

    with torch.no_grad():
        for data, labels in test_loader:
            data = data.to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
            labels_list.append(labels)

    latents = torch.cat(latents, dim=0).numpy()
    labels_list = torch.cat(labels_list, dim=0).numpy()

    # Reduce to 2D
    if method == 'tsne':
        print("Running t-SNE (this may take a minute)...")
        reducer = TSNE(n_components=2, random_state=42, perplexity=30)
        latents_2d = reducer.fit_transform(latents)
    else:  # PCA
        print("Running PCA...")
        reducer = PCA(n_components=2)
        latents_2d = reducer.fit_transform(latents)
        print(f"Explained variance: {reducer.explained_variance_ratio_.sum():.2%}")

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(latents_2d[:, 0], latents_2d[:, 1], c=labels_list,
                          cmap='tab10', alpha=0.6, s=10)
    plt.colorbar(scatter, label='Digit')
    plt.xlabel('Dimension 1')
    plt.ylabel('Dimension 2')
    plt.title(f'VAE 32D Latent Space ({method.upper()} projection)')
    plt.grid(True, alpha=0.3)
    plt.show()

# Visualize the 32D VAE latent space
plot_latent_space_highdim(vae, test_loader, method='pca')  # Fast
plot_latent_space_highdim(vae, test_loader, method='tsne')  # Better clustering, slower


Or Train a 2D VAE for direct visualization

Alternatively, train a VAE with 2D latent space to visualize directly and see the manifold.

In [ ]:
# Train a 2D VAE for visualization # or you can use K-mean to project high-dim to 2-dim
vae_2d = VAE(latent_dim=2)
print("Training 2D VAE (may have lower quality due to low dimensionality)...")
vae_2d_losses = train_vae(vae_2d, train_loader, epochs=30, lr=1e-3)

In [ ]:
def plot_latent_space(model, test_loader):
    """Plot the 2D latent space colored by digit class"""
    model.eval()

    latents = []
    labels_list = []

    with torch.no_grad():
        for data, labels in test_loader:
            data = data.to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
            labels_list.append(labels)

    latents = torch.cat(latents, dim=0).numpy()
    labels_list = torch.cat(labels_list, dim=0).numpy()

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(latents[:, 0], latents[:, 1], c=labels_list,
                          cmap='tab10', alpha=0.6, s=10)
    plt.colorbar(scatter, label='Digit')
    plt.xlabel('Latent Dimension 1')
    plt.ylabel('Latent Dimension 2')
    plt.title('VAE 2D Latent Space (colored by digit)')
    plt.grid(True, alpha=0.3)
    plt.show()

plot_latent_space(vae_2d, test_loader)

In [ ]:
def plot_latent_manifold(model, n=20, digit_size=28):
    """Generate images by sampling the 2D latent space uniformly"""
    model.eval()

    # Create grid of latent values
    grid_x = np.linspace(-3, 3, n)
    grid_y = np.linspace(-3, 3, n)

    figure = np.zeros((digit_size * n, digit_size * n))

    with torch.no_grad():
        for i, yi in enumerate(grid_y):
            for j, xi in enumerate(grid_x):
                z = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
                digit = model.decode(z).cpu().numpy().reshape(digit_size, digit_size)
                figure[i * digit_size: (i + 1) * digit_size,
                       j * digit_size: (j + 1) * digit_size] = digit

    plt.figure(figsize=(12, 12))
    plt.imshow(figure, cmap='gray')
    plt.xlabel('z[0]')
    plt.ylabel('z[1]')
    plt.title('VAE Latent Space Manifold')
    plt.axis('off')
    plt.show()

plot_latent_manifold(vae_2d, n=15)

---

# Part 3: Generative Adversarial Network (GAN) ⚔️

## 3.1 Background

A **GAN** consists of two networks that compete:

- **Generator (G)**: Takes random noise and generates fake images
- **Discriminator (D)**: Tries to distinguish real images from fake ones

They are trained in a minimax game:

$$\min_G \max_D \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

## 3.2 Training Process

1. Train D on real images (label = 1) and fake images (label = 0)
2. Train G to fool D (want D(G(z)) = 1)

---

## Exercise 3.1: Implement the Generator

The generator takes random noise and produces images.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, hidden_dim=256, output_dim=784):
        super(Generator, self).__init__()

        # TODO: Define the generator network
        # Architecture: latent_dim → 128 → 256 → 512 → output_dim
        # Use LeakyReLU activation (slope=0.2) for hidden layers
        # Use Tanh activation at output (to get values in [-1, 1])

        self.model = nn.Sequential(
            # TODO: Layer 1: latent_dim → 128, LeakyReLU
            # TODO: Layer 2: 128 → 256, LeakyReLU
            # TODO: Layer 3: 256 → 512, LeakyReLU
            # TODO: Layer 4: 512 → output_dim, Tanh
        )

    def forward(self, z):
        # TODO: Pass z through the model and reshape to image
        # Output shape should be (batch_size, 1, 28, 28)

        pass  # TODO: Implement


# Test generator
test_gen = Generator()
test_z = torch.randn(4, 100)
try:
    test_fake = test_gen(test_z)
    assert test_fake.shape == (4, 1, 28, 28), f"Expected (4, 1, 28, 28), got {test_fake.shape}"
    print("✅ Generator test passed! Output shape:", test_fake.shape)
except Exception as e:
    print(f"❌ Generator test failed: {e}")

## Exercise 3.2: Implement the Discriminator

The discriminator takes an image and outputs a probability of it being real.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256):
        super(Discriminator, self).__init__()

        # TODO: Define the discriminator network
        # Architecture: input_dim → 512 → 256 → 128 → 1
        # Use LeakyReLU activation (slope=0.2) for hidden layers
        # Use Sigmoid at output to get probability
        # Add Dropout (0.3) after each LeakyReLU for stability

        self.model = nn.Sequential(
            # TODO: Layer 1: input_dim → 512, LeakyReLU, Dropout
            # TODO: Layer 2: 512 → 256, LeakyReLU, Dropout
            # TODO: Layer 3: 256 → 128, LeakyReLU, Dropout
            # TODO: Layer 4: 128 → 1, Sigmoid
        )

    def forward(self, img):
        # TODO: Flatten the image and pass through the model
        # Input: (batch_size, 1, 28, 28)
        # Output: (batch_size, 1) - probability of being real

        pass  # TODO: Implement


# Test discriminator
test_disc = Discriminator()
test_img = torch.randn(4, 1, 28, 28)
try:
    test_prob = test_disc(test_img)
    assert test_prob.shape == (4, 1), f"Expected (4, 1), got {test_prob.shape}"
    assert test_prob.min() >= 0 and test_prob.max() <= 1, "Output should be probability [0, 1]"
    print("✅ Discriminator test passed! Output shape:", test_prob.shape)
except Exception as e:
    print(f"❌ Discriminator test failed: {e}")

## Exercise 3.3: GAN Training Loop

This is the most complex part. We need to:
1. Train the discriminator to distinguish real from fake
2. Train the generator to fool the discriminator

**Important:** For GAN training, images should be normalized to [-1, 1] to match Tanh output.

In [ ]:
# Create a new data loader with images normalized to [-1, 1]
transform_gan = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

train_dataset_gan = datasets.MNIST(root='./data', train=True, download=True, transform=transform_gan)
train_loader_gan = DataLoader(train_dataset_gan, batch_size=128, shuffle=True)

In [ ]:
def train_gan(generator, discriminator, train_loader, epochs=30, lr=0.0002, latent_dim=100):
    """
    Train the GAN.

    Args:
        generator: Generator network
        discriminator: Discriminator network
        train_loader: DataLoader with training data
        epochs: Number of epochs
        lr: Learning rate
        latent_dim: Dimension of latent noise vector
    """
    generator = generator.to(device)
    discriminator = discriminator.to(device)

    # TODO: Create optimizers for both networks
    # Use Adam with betas=(0.5, 0.999) for better GAN training
    optimizer_G = None  # TODO: Optimizer for generator
    optimizer_D = None  # TODO: Optimizer for discriminator

    # Loss function: Binary Cross Entropy
    criterion = nn.BCELoss()

    # Labels
    real_label = 1.0
    fake_label = 0.0

    g_losses = []
    d_losses = []

    # Fixed noise for visualization
    fixed_noise = torch.randn(16, latent_dim, device=device)

    for epoch in range(epochs):
        epoch_d_loss = 0
        epoch_g_loss = 0

        for batch_idx, (real_images, _) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)

            # Create labels
            real_labels = torch.full((batch_size, 1), real_label, device=device)
            fake_labels = torch.full((batch_size, 1), fake_label, device=device)

            # ================== Train Discriminator ==================
            # TODO: Implement discriminator training
            # Step 1: Zero gradients for discriminator
            # Step 2: Forward pass with REAL images → D(real)
            # Step 3: Compute loss for real images (should be close to 1)
            # Step 4: Generate FAKE images: z → G(z)
            # Step 5: Forward pass with fake images → D(G(z))
            #         Important: Use .detach() on fake images to not update G
            # Step 6: Compute loss for fake images (should be close to 0)
            # Step 7: Total D loss = real_loss + fake_loss
            # Step 8: Backward pass and update D

            optimizer_D.zero_grad()

            # TODO: Your code here for discriminator training
            d_loss = None  # TODO: Compute total discriminator loss

            # ================== Train Generator ==================
            # TODO: Implement generator training
            # Step 1: Zero gradients for generator
            # Step 2: Generate fake images (can reuse z from above or sample new)
            # Step 3: Forward pass: D(G(z))
            # Step 4: Compute loss (want D to output 1 for fake images)
            #         So use real_labels as target!
            # Step 5: Backward pass and update G

            optimizer_G.zero_grad()

            # TODO: Your code here for generator training
            g_loss = None  # TODO: Compute generator loss

            if d_loss is not None and g_loss is not None:
                epoch_d_loss += d_loss.item()
                epoch_g_loss += g_loss.item()

        n_batches = len(train_loader)
        d_losses.append(epoch_d_loss / n_batches)
        g_losses.append(epoch_g_loss / n_batches)

        print(f'Epoch {epoch+1}: D_loss={epoch_d_loss/n_batches:.4f}, G_loss={epoch_g_loss/n_batches:.4f}')

        # Visualize progress every 5 epochs
        if (epoch + 1) % 5 == 0:
            generator.eval()
            with torch.no_grad():
                fake_samples = generator(fixed_noise)

            fig, axes = plt.subplots(2, 8, figsize=(16, 4))
            for i, ax in enumerate(axes.flat):
                img = fake_samples[i].cpu().squeeze()
                ax.imshow(img * 0.5 + 0.5, cmap='gray')  # Denormalize
                ax.axis('off')
            plt.suptitle(f'Generated Samples - Epoch {epoch+1}')
            plt.show()
            generator.train()

    return g_losses, d_losses

In [ ]:
# Create and train GAN
latent_dim = 100
generator = Generator(latent_dim=latent_dim)
discriminator = Discriminator()

g_losses, d_losses = train_gan(generator, discriminator, train_loader_gan, epochs=30, latent_dim=latent_dim) # one can increase the epochs

In [ ]:
# Plot training losses
plt.figure(figsize=(12, 4))
plt.plot(g_losses, label='Generator Loss', linewidth=2)
plt.plot(d_losses, label='Discriminator Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Losses')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Generate Final Samples

In [ ]:
def generate_gan_samples(generator, n_samples=25, latent_dim=100):
    """Generate samples from trained GAN"""
    generator.eval()

    with torch.no_grad():
        z = torch.randn(n_samples, latent_dim, device=device)
        samples = generator(z)

    # Plot
    n_rows = int(np.sqrt(n_samples))
    n_cols = n_samples // n_rows

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 12))

    for i, ax in enumerate(axes.flat):
        img = samples[i].cpu().squeeze()
        ax.imshow(img * 0.5 + 0.5, cmap='gray')  # Denormalize from [-1, 1] to [0, 1]
        ax.axis('off')

    plt.suptitle('GAN Generated Samples', fontsize=14)
    plt.tight_layout()
    plt.show()

generate_gan_samples(generator, n_samples=25, latent_dim=latent_dim)

## Latent Space Interpolation

One interesting property of GANs is smooth interpolation in latent space.

In [ ]:
def interpolate_latent(generator, latent_dim=100, n_steps=10):
    """Interpolate between two random points in latent space"""
    generator.eval()

    # Sample two random points
    z1 = torch.randn(1, latent_dim, device=device)
    z2 = torch.randn(1, latent_dim, device=device)

    # Linear interpolation
    alphas = torch.linspace(0, 1, n_steps)

    fig, axes = plt.subplots(1, n_steps, figsize=(20, 2))

    with torch.no_grad():
        for i, alpha in enumerate(alphas):
            z = (1 - alpha) * z1 + alpha * z2
            img = generator(z).cpu().squeeze()
            axes[i].imshow(img * 0.5 + 0.5, cmap='gray')
            axes[i].axis('off')
            axes[i].set_title(f'α={alpha:.1f}')

    plt.suptitle('Latent Space Interpolation', fontsize=14)
    plt.tight_layout()
    plt.show()

interpolate_latent(generator, latent_dim=latent_dim, n_steps=10)

1. What happens if the discriminator becomes too strong? What about if the generator becomes too strong?
2. Why do we use LeakyReLU instead of ReLU in GANs?
3. What is "mode collapse"? Did you observe it during training?
4. Compare the quality of samples from VAE vs GAN. Which looks sharper? Why?


---

If you finish early, you can try other extensions or adapt to your research:

1. Implement a **Convolutional VAE** (use Conv2d layers instead of Linear)
2. Implement **Conditional GAN** (generate specific digits by conditioning on labels)
3. Try **DCGAN** architecture (Deep Convolutional GAN) for better image quality
4. Implement **β-VAE** (add a β coefficient to the KL divergence term)

---